# SafeInstall — Installer Malware Classifier

Trains the model that powers the **ML Model** finding in the `safe` CLI. Features come
from the Rust extractor (`safe features`), so the notebook and the shipped binary see an
identical representation. The exported model is pure Rust (`src/ml_model.rs`) — no runtime
dependency at inference time.

This notebook mirrors `ml/train.py`; run `python ml/build_dataset.py` first to produce
`data/features.csv`.

In [ ]:
import pathlib, matplotlib.pyplot as plt, numpy as np, pandas as pd, shap
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                             precision_score, recall_score, f1_score, classification_report)
SEED=1706
DATA=pathlib.Path('..')/'data'
df=pd.read_csv(DATA/'features.csv')
print(df.shape); df['label'].value_counts()

## 1. Stratified train/test split

In [ ]:
y=df['label'].to_numpy(); X=df.drop(columns=['label'])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,stratify=y,random_state=SEED)
len(Xtr),len(Xte)

## 2. RandomForest vs GradientBoosting
GB is compared for reference; only RandomForest exports to dependency-free Rust via m2cgen, so it is the deployed model.

In [ ]:
models={'RandomForest':RandomForestClassifier(n_estimators=120,max_depth=9,min_samples_leaf=2,
                          class_weight='balanced',random_state=SEED,n_jobs=-1),
        'GradientBoosting':GradientBoostingClassifier(n_estimators=120,max_depth=3,random_state=SEED)}
for n,m in models.items():
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; pred=(p>=0.5).astype(int)
    print(f"{n:18s} P={precision_score(yte,pred):.3f} R={recall_score(yte,pred):.3f} F1={f1_score(yte,pred):.3f}")
clf=models['RandomForest']; proba=clf.predict_proba(Xte)[:,1]

## 3. Threshold for a low false-positive rate

In [ ]:
def pick(y,p,max_fpr=0.02):
    best_t,best_r=0.5,-1
    for t in np.linspace(0.1,0.95,86):
        pred=(p>=t).astype(int); fp=((pred==1)&(y==0)).sum(); tn=((pred==0)&(y==0)).sum()
        r=recall_score(y,pred) if pred.sum() else 0
        if fp/max(tn+fp,1)<=max_fpr and r>best_r: best_r,best_t=r,t
    return round(float(best_t),3)
THRESH=pick(yte,proba); pred=(proba>=THRESH).astype(int)
print('threshold',THRESH); print(classification_report(yte,pred,target_names=['benign','malicious']))

## 4. Confusion matrix and ROC

In [ ]:
cm=confusion_matrix(yte,pred)
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].imshow(cm,cmap='BuGn'); ax[0].set_title('Confusion matrix')
ax[0].set_xticks([0,1],['benign','mal']); ax[0].set_yticks([0,1],['benign','mal'])
for i in range(2):
    for j in range(2): ax[0].text(j,i,cm[i,j],ha='center',va='center')
fpr,tpr,_=roc_curve(yte,proba)
ax[1].plot(fpr,tpr,label=f'AUC={auc(fpr,tpr):.3f}'); ax[1].plot([0,1],[0,1],'--')
ax[1].set_title('ROC'); ax[1].legend(); plt.tight_layout(); plt.show()

## 5. SHAP feature attribution

In [ ]:
expl=shap.TreeExplainer(clf); sv=expl.shap_values(Xtr.sample(300,random_state=SEED))
vals=sv[1] if isinstance(sv,list) else (sv[...,1] if sv.ndim==3 else sv)
shap.summary_plot(vals,Xtr.sample(300,random_state=SEED),max_display=15)

## 6. Export to Rust
See `ml/train.py:export_rust` — writes `src/ml_model.rs` via `m2cgen.export_to_rust` and verifies Python/Rust probability parity (max abs diff ~5e-11).

In [ ]:
import m2cgen as m2c
rust=m2c.export_to_rust(clf)
print(rust.splitlines()[0], '...', f'{len(rust.splitlines())} lines generated')